In [1]:
import numpy as np

import torch
from math import e

In [2]:
X = np.array([160, 170, 180, 190])
y = np.array([0, 0, 1, 1])

X, y

(array([160, 170, 180, 190]), array([0, 0, 1, 1]))

In [3]:
X_mean = np.mean(X)
X_std = np.std(X)

X_norm = (X-X_mean) / X_std

X_mean, X_std, X_norm

(np.float64(175.0),
 np.float64(11.180339887498949),
 array([-1.34164079, -0.4472136 ,  0.4472136 ,  1.34164079]))

In [4]:
X_norm_tensor = torch.tensor(X_norm, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

X_norm_tensor, y_tensor

(tensor([-1.3416, -0.4472,  0.4472,  1.3416]), tensor([0., 0., 1., 1.]))

In [5]:
a = torch.tensor(0.1, dtype=torch.float32, requires_grad=True)
b = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)

a.item(), b.item()

(0.10000000149011612, 0.0)

In [ ]:
def sigmoid(H):
    return 1 / (1+ e **(-H))



In [7]:
# 학습 전 예측 확인
H = a.item() * X_norm + b.item()

z = sigmoid(H)
H, z

(array([-0.13416408, -0.04472136,  0.04472136,  0.13416408]),
 array([0.4665092 , 0.48882152, 0.51117848, 0.5334908 ]))

In [8]:
# 학습 전 costs 확인
epsilon = 1e-7
z_safe = np.clip(z, epsilon, 1 - epsilon)

costs = -(y * np.log(z_safe) + (1-y) * np.log(1-z_safe))
mean_cost= np.mean(costs)
costs, mean_cost

(array([0.62831345, 0.67103648, 0.67103648, 0.62831345]),
 np.float64(0.6496749672265923))

In [9]:
learning_rate = 0.1
epochs = 1000

for epoch in range(epochs):
    H = a * X_norm_tensor + b
    z = torch.sigmoid(H)

    z_safe = torch.clamp(z, epsilon, 1 - epsilon)
    costs = - (y_tensor * torch.log(z_safe) + (1 - y_tensor) * torch.log(1 - z_safe))
    mean_cost = torch.mean(costs)

    # grad_a = mean((z-y) * x_norm)
    # grad_b = mean(z-y)
    mean_cost.backward()

    grad_a = a.grad.item()
    grad_b = b.grad.item()

    with torch.no_grad():
        # 경사하강법 a, b 업데이트
        a -= learning_rate * a.grad
        b -= learning_rate * b.grad

    # 다음 epoch를 위해 gradient초기화
    # zero_() 에서 _ : 새 tensor를 만드는게 아니라 기존 값을 0으로 바꾼다
    a.grad.zero_()
    b.grad.zero_()

    if epoch % 100 == 0 or epoch == epochs - 1:
        print(
            f'Epoch {epoch}, '
            f'평균 비용(Cost): {mean_cost.item():.6f}, '
            f'grad_a: {grad_a:.6f}, '
            f'grad_b: {grad_b:.6f}, '
            f'a: {a.item():.6f}, '
            f'b: {b.item():.6f}'
        )

Epoch 0, 평균 비용(Cost): 0.649675, grad_a: -0.422248, grad_b: 0.000000, a: 0.142225, b: -0.000000
Epoch 100, 평균 비용(Cost): 0.198225, grad_a: -0.103535, grad_b: 0.000000, a: 2.069146, b: 0.000000
Epoch 200, 평균 비용(Cost): 0.133789, grad_a: -0.063040, grad_b: -0.000000, a: 2.860633, b: 0.000000
Epoch 300, 평균 비용(Cost): 0.104196, grad_a: -0.047123, grad_b: -0.000000, a: 3.401512, b: 0.000000
Epoch 400, 평균 비용(Cost): 0.086192, grad_a: -0.038251, grad_b: 0.000000, a: 3.824389, b: 0.000000
Epoch 500, 평균 비용(Cost): 0.073786, grad_a: -0.032440, grad_b: -0.000000, a: 4.175776, b: 0.000000
Epoch 600, 평균 비용(Cost): 0.064613, grad_a: -0.028273, grad_b: 0.000000, a: 4.478100, b: 0.000000
Epoch 700, 평균 비용(Cost): 0.057512, grad_a: -0.025108, grad_b: 0.000000, a: 4.744184, b: 0.000000
Epoch 800, 평균 비용(Cost): 0.051834, grad_a: -0.022607, grad_b: 0.000000, a: 4.982181, b: 0.000000
Epoch 900, 평균 비용(Cost): 0.047181, grad_a: -0.020573, grad_b: -0.000000, a: 5.197650, b: 0.000000
Epoch 999, 평균 비용(Cost): 0.043330, gra

In [10]:
print(f'\n학습 완료 후의 최적값: a = {a.item()}, b ={b.item()}')


학습 완료 후의 최적값: a = 5.3927106857299805, b =8.616920865733846e-08


In [11]:
# 새로운 입력값 예측
input_height = 185
# 정규화
input_norm = (input_height - X_mean) / X_std

with torch.no_grad():
    # tensor 변환
    input_norm_tensor = torch.tensor(input_norm, dtype=torch.float32)
    H_input = a * input_norm_tensor + b
    probability = torch.sigmoid(H_input)

print(f'\n키가 {input_height}cm인 사람의 농구선수 확률: {probability.item():.4f}')

pred = 1 if probability.item() >= 0.5 else 0
if pred == 1:
    print('판별 결과: 농구선수입니다.')
else:
    print('판별 결과: 농구선수가 아닙니다.')


키가 185cm인 사람의 농구선수 확률: 0.9920
판별 결과: 농구선수입니다.
